# Patrón State (Estado)

## 1) ¿Qué problema resuelve?
Cuando un objeto cambia de comportamiento dependiendo de su **estado interno** y terminas con lógica tipo:

- `if state == A: ...`
- `elif state == B: ...`
- `elif state == C: ...`

Este enfoque:
- escala mal (explosión de condicionales),
- se vuelve difícil de mantener,
- produce transiciones inválidas o inconsistentes,
- y mezcla “reglas de transición” con “acciones”.

**State** encapsula el comportamiento por estado en objetos separados.

---

## 2) Idea central
“**El objeto delega su comportamiento al objeto estado actual**”.

En lugar de preguntar:
- “¿en qué estado estoy?” y luego decidir con if/else,

haces:
- `current_state.handle(context)`

y cada estado sabe:
- qué acciones permite,
- a qué estados puede transicionar,
- y qué ocurre en cada acción.

---

## 3) Componentes
- **Context**: el objeto principal (ej. `Order`, `Task`, `TrafficLight`)
- **State (interfaz/ABC)**: define operaciones que cambian según estado (ej. `pay()`, `ship()`, `cancel()`)
- **ConcreteState**: implementaciones concretas por estado (ej. `CreatedState`, `PaidState`, `ShippedState`)

---

## 4) Cuándo usarlo (señales)
- Reglas de comportamiento cambian por estado: “cancel” hace cosas distintas según estado.
- Existen transiciones válidas e inválidas que quieres controlar formalmente.
- El objeto se modela naturalmente como **máquina de estados**.
- Hay muchos if/else basados en `status`.

---

## 5) Beneficios
- Elimina condicionales: comportamiento vive en estados.
- Transiciones explícitas: cada estado controla lo permitido.
- Facilita extensibilidad: agregar un estado nuevo no revienta el core.
- Modelado más cercano a dominio (finite state machine).

---

## 6) Trade-offs
- Aumenta número de clases/archivos.
- Requiere disciplina para mantener coherencia entre estados y transiciones.
- Si el flujo es pequeño y estable, puede ser overkill.

---

## 7) Diferencia con Strategy
- **Strategy**: cambia un *algoritmo* (cómo calculo algo).
- **State**: cambia *comportamiento completo* según el estado interno y transiciones.

En State, el “context” suele cambiar la estrategia/estado dinámicamente.

# Ejercicio dummy: Logica de un Semaforo

Estados: Rojo (parar), Amarillo (disminuir velocidad), Verde (Seguir)

-> next(): Rojo -> Verde -> Amarillo -> Rojo -> .... movimiento de los estados del semaforo

In [ ]:
from enum import Enum

class Luz(Enum):
    ROJO: "ROJO"
    VERDE: "VERDE"
    AMARILLO: "AMARILLO"

In [ ]:
class Semaforo:
    def __init__(self):
        self.state = Luz.ROJO
    
    def next(self):
        if self.state == Luz.ROJO:
            self.state = Luz.VERDE
        elif self.state == Luz.VERDE:
            self.state = Luz.AMARILLO
        else:
            self.state = Luz.ROJO
    
    def action(self):
        if self.state == Luz.ROJO:
            return 'DETENERSE'
        elif self.state == Luz.VERDE:
            return 'SEGUIR'
        else:
            return 'DISMINUIR VELOCIDAD'

In [1]:
from abc import ABC, abstractmethod

class EstadoSemaforo(ABC):
    @abstractmethod
    def next(self, contexto: "LuzSemaforo") -> None:
        pass

    @abstractmethod
    def action(self) -> str:
        pass

In [10]:
# Contexto: delega en el estado actual y centraliza el set_state()
class LuzSemaforo:
    def __init__(self):
        # Estado inicial
        self._state: EstadoSemaforo = EstadoRojo()
    
    def set_state(self, estado: EstadoSemaforo) -> None:
        self._state = estado
    
    def next(self) -> None:
        self._state.next(self)
    
    def action(self) -> str:
        return self._state.action()
    

In [2]:
class EstadoRojo(EstadoSemaforo):
    def next(self, contexto: 'LuzSemaforo') -> None:
        contexto.set_state(EstadoVerde())
    def action(self) -> str:
        return 'Detenerse'

In [3]:
class EstadoVerde(EstadoSemaforo):
    def next(self, contexto: 'LuzSemaforo') -> None:
        contexto.set_state(EstadoAmarillo())
    def action(self) -> str:
        return 'Seguir'

In [4]:
class EstadoAmarillo(EstadoSemaforo):
    def next(self, contexto: 'LuzSemaforo') -> None:
        contexto.set_state(EstadoRojo())
    def action (self) -> str:
        return 'Disminuir Velocidad'

In [12]:
semaforo = LuzSemaforo()

for _ in range(10):
    print(semaforo.action())
    semaforo.next()

Detenerse
Seguir
Disminuir Velocidad
Detenerse
Seguir
Disminuir Velocidad
Detenerse
Seguir
Disminuir Velocidad
Detenerse


In [ ]:
# Implementen un estado mas:  amarillo -> Rojo parpadeante -> rojo -> verde -> amarillo